# Chapter 4: Discrete Behavior Cloning

This Colab follows the Chapter 4 manuscript from the multimodal regression trap through a trained, discrete SO-101 action policy. A GPU runtime is strongly recommended for the real backbone cells.

In [ ]:
# Colab setup: install the three public chapter packages from GitHub.
import subprocess
import sys

if 'google.colab' in sys.modules:
    organization = 'Large-Robotics-Models-From-Scratch'
    chapter4_requirement = (
        f'lrm-ch04[data] @ git+https://github.com/{organization}/'
        'lrm-code-chapter-4.git@codex/fix-vla-backbone-loading'
    )
    requirements = [
        f'lrm-ch02[data] @ git+https://github.com/{organization}/lrm-code-chapter-2.git@main',
        f'lrm-ch03 @ git+https://github.com/{organization}/lrm-code-chapter-3.git@main',
        chapter4_requirement,
    ]
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', *requirements],
        text=True, capture_output=True,
    )
    if result.returncode:
        detail = '\n'.join(
            part for part in (result.stdout, result.stderr) if part
        )
        raise RuntimeError(
            f'Chapter package installation failed:\n{detail}'
        )
    # lrm-ch04 is still version 0.1.0 while this branch is under
    # development, so pip may retain an older 0.1.0 wheel in a reused
    # runtime. Refresh only this small package; keep resolved dependencies.
    refresh = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet',
         '--force-reinstall', '--no-deps', chapter4_requirement],
        text=True, capture_output=True,
    )
    if refresh.returncode:
        detail = '\n'.join(
            part for part in (refresh.stdout, refresh.stderr) if part
        )
        raise RuntimeError(
            f'Chapter 4 branch refresh failed:\n{detail}'
        )
    api_probe = subprocess.run(
        [sys.executable, '-c',
         'from ch04.decoding import decode_action_chunk, '
         'sample_action_grids'],
        text=True, capture_output=True,
    )
    if api_probe.returncode:
        detail = '\n'.join(
            part for part in (api_probe.stdout, api_probe.stderr)
            if part
        )
        raise RuntimeError(
            f'Chapter 4 API verification failed:\n{detail}'
        )
    loaded_ch04 = [
        name for name in sys.modules
        if name == 'ch04' or name.startswith('ch04.')
    ]
    for name in loaded_ch04:
        sys.modules.pop(name, None)
    if loaded_ch04:
        print('Cleared cached Chapter 4 modules; rerun from here onward.')
    # LeRobot may upgrade Colab's preinstalled torch without upgrading
    # its optional torchaudio wheel. Transformers detects torchaudio by
    # package presence, then imports the incompatible binary while loading
    # SigLIP. Probe in a child process so a failed import cannot taint this
    # kernel; Chapters 2-4 do not use audio, so remove only a broken wheel.
    audio_probe = subprocess.run(
        [sys.executable, '-c', 'import torch, torchaudio'],
        text=True, capture_output=True,
    )
    if audio_probe.returncode:
        uninstall = subprocess.run(
            [sys.executable, '-m', 'pip', 'uninstall', '--yes',
             'torchaudio'],
            text=True, capture_output=True,
        )
        if uninstall.returncode:
            detail = '\n'.join(
                part for part in (uninstall.stdout, uninstall.stderr)
                if part
            )
            raise RuntimeError(
                f'Could not remove incompatible torchaudio:\n{detail}'
            )
        print('Removed an incompatible optional torchaudio wheel.')
    print('Installed Chapter 2, Chapter 3, and Chapter 4 packages.')

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

from ch04.constants import ACTION_BINS, ACTION_DIM, ACTION_HORIZON

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print(f'label grid: H={ACTION_HORIZON}, D={ACTION_DIM}, B={ACTION_BINS}')

## 4.2 The multimodal regression trap

The observation contains no clue about which of two equally valid expert modes was chosen. MSE therefore learns the conditional mean: zero, where the demonstrations have almost no density.

In [ ]:
from ch04.exercises import make_bimodal_actions, train_mse_baseline

observations, expert_actions = make_bimodal_actions()
mse_model, mse_history = train_mse_baseline()
with torch.no_grad():
    collapsed = mse_model(torch.zeros(1, 1)).item()
print(f'MSE prediction: {collapsed:+.3f} (expert modes are -1 and +1)')

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(expert_actions.numpy(), bins=40)
axes[0].axvline(collapsed, color='crimson', label='MSE prediction')
axes[0].legend(); axes[0].set_xlabel('action')
axes[1].plot(mse_history); axes[1].set(xlabel='step', ylabel='MSE')
plt.show()

## 4.3 Fit the action tokenizer

We compute normalization statistics from training episodes only, then fit per-joint q01/q99 limits in that normalized space. The helper reads state and action columns directly from Arrow, so the full fit does not decode the camera videos.

In [ ]:
from ch04 import ActionTokenizer
from ch04.data import (DEFAULT_DATASET_ID, collect_normalized_actions,
                         make_chunked_dataloaders)

train_loader, validation_loader, stats = make_chunked_dataloaders(
    DEFAULT_DATASET_ID, batch_size=4, validation_fraction=0.1)
# None uses every training frame through the fast Arrow action column.
# Set an integer only for a bounded loader-based smoke fit.
TOKENIZER_FIT_BATCHES = None
normalized_actions = collect_normalized_actions(
    train_loader, stats, max_batches=TOKENIZER_FIT_BATCHES)
tokenizer = ActionTokenizer.fit(normalized_actions)

example = normalized_actions[0]
bins = tokenizer.encode(example)
decoded = tokenizer.decode(bins)
print('bins:   ', bins)
print('AR action embedding ids:', bins, '(separate 256-entry table)')
print('max normalized round-trip error:', np.abs(decoded-example).max())
print('tokenizer fit batches:', TOKENIZER_FIT_BATCHES or 'all')
print('train episodes:', train_loader.dataset.episodes)
print('validation episodes:', validation_loader.dataset.episodes)

In [ ]:
# A chunk has H vector-valued action positions and H x D labels.
demo_grid = torch.arange(ACTION_HORIZON * ACTION_DIM).reshape(
    1, ACTION_HORIZON, ACTION_DIM)
print('target grid:', tuple(demo_grid.shape))
print('action positions:', ACTION_HORIZON)
print('first two vectors:', demo_grid[0, :2].tolist())

## 4.4 Build the three action heads

The factorized head is the one-shot baseline, the autoregressive head is the exact chain-rule model, and the bidirectional parallel head is the manuscript's one-pass path. Each experiment starts from a fresh Chapter 3 backbone, so training one design cannot improve the next design's starting point.

In [ ]:
from ch03 import VLABackbone
from ch04 import (AutoregressiveActionHead, FactorizedActionHead,
                  ParallelDecodeActionHead)


def build_action_head(name):
    backbone = VLABackbone().to(device)
    backbone.language_backbone.set_attn_implementation('eager')
    builders = {
        'factorized': lambda: FactorizedActionHead(),
        'autoregressive': lambda: AutoregressiveActionHead(backbone),
        'parallel': lambda: ParallelDecodeActionHead(backbone),
    }
    if name not in builders:
        raise ValueError(f'unknown action head: {name}')
    return backbone, builders[name]().to(device)

print('experiment order: factorized -> autoregressive -> parallel')

## 4.5 Shared training, visualization, and evaluation

Only the logits path varies across architectures. Factorized and parallel logits come directly from the observation; autoregressive training and marginal visualization use teacher forcing. Complete autoregressive samples and decoded chunks still use causal KV-cached generation.

### 4.5.1 Factorized head

Define the shared runner, then execute the fastest baseline first.

In [ ]:
import itertools

from ch04.data import action_targets, prepare_batch
from ch04.decoding import (decode_action_chunk, evaluate_open_loop,
                             evaluation_mode, sample_action_grids)
from ch04.diagnostics import (plot_action_distribution,
                              plot_chunk_comparison,
                              plot_joint_support, temporal_jitter,
                              within_expert_support)
from ch04.losses import masked_token_cross_entropy
from ch04.train import action_head_logits, train_action_head


def run_head_experiment(
        name, steps=10, samples=64, eval_batches=1, seed=7):
    torch.manual_seed(seed)
    np.random.seed(seed)
    backbone, head = build_action_head(name)
    history = train_action_head(
        head, backbone, train_loader, stats, tokenizer, device,
        total_steps=steps, warmup_steps=min(5, steps - 1),
        log_every=max(1, steps // 20), checkpoint_every=steps,
        checkpoint_dir=f'/content/ch04-checkpoints/{name}')

    batch = next(iter(validation_loader))
    model_inputs = prepare_batch(batch, stats, device, backbone)
    target_bins, token_pad = action_targets(
        batch, stats, tokenizer, device)
    with torch.no_grad(), evaluation_mode(backbone), evaluation_mode(head):
        logits = action_head_logits(
            head, backbone, model_inputs, target_bins)
        validation_ce = masked_token_cross_entropy(
            logits, target_bins, token_pad).item()

    sampled_grids = sample_action_grids(
        head, backbone, model_inputs, n_samples=samples)
    prediction = decode_action_chunk(
        head, backbone, model_inputs, tokenizer, stats,
        strategy='argmax').cpu()
    metrics = evaluate_open_loop(
        head, itertools.islice(validation_loader, eval_batches),
        tokenizer, stats, backbone, device)

    expert = torch.as_tensor(batch['action']).float()
    expert_pairs = target_bins[:, 0, [4, 5]].cpu().numpy()
    draws = sampled_grids[:, 0, [4, 5]].cpu().numpy()
    supported = within_expert_support(draws, expert_pairs, 8.0)
    jitter = np.mean([
        temporal_jitter(grid) for grid in sampled_grids.cpu().numpy()
    ])

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    axes[0].plot([row['step'] for row in history],
                 [row['loss'] for row in history], label='CE')
    axes[0].plot([row['step'] for row in history],
                 [row['entropy'] for row in history], label='entropy')
    axes[0].set(xlabel='step', title=f'{name}: training')
    axes[0].legend()
    probabilities = logits[0, 0, 4].softmax(-1).cpu().numpy()
    plot_action_distribution(probabilities, ax=axes[1])
    axes[1].set_title('timestep 0, control 4')
    plot_joint_support(
        expert_pairs, draws, supported, 8.0, ax=axes[2])
    axes[2].set_title(f'{name}: complete-grid draws')
    fig.tight_layout()
    plt.show()
    plot_chunk_comparison(
        prediction[0].numpy(), expert[0].numpy())
    plt.suptitle(f'{name}: held-out action chunk', y=1.01)
    plt.show()

    mae_std = metrics['mae_in_standard_deviations'].nanmean().item()
    print(f'{name}: validation CE={validation_ce:.3f}, '
          f'MAE/std={mae_std:.3f}, sampled jitter={jitter:.3f}')
    logits = logits.cpu()
    sampled_grids = sampled_grids.cpu()
    head.cpu()
    backbone.cpu()
    del model_inputs, target_bins, token_pad
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {
        'head': head, 'backbone': backbone, 'history': history,
        'validation_ce': validation_ce, 'mae_std': mae_std,
        'jitter': float(jitter), 'prediction': prediction,
        'logits': logits, 'samples': sampled_grids,
    }

In [ ]:
# Factorized head first: fastest end-to-end pipeline check.
TRAIN_STEPS = 10  # use 20_000 for a full, comparable run
results = {}
results['factorized'] = run_head_experiment(
    'factorized', steps=TRAIN_STEPS)

### 4.5.2 Autoregressive head

Training remains one teacher-forced pass. Sampling and argmax decoding perform 96 causal steps with a KV cache, so this cell takes longer.

In [ ]:
results['autoregressive'] = run_head_experiment(
    'autoregressive', steps=TRAIN_STEPS, samples=32)

### 4.5.3 Bidirectional parallel head

The manuscript's main path adds 16 interacting action positions and predicts the full chunk in one pass.

In [ ]:
results['parallel'] = run_head_experiment(
    'parallel', steps=TRAIN_STEPS)

## 4.6 Compare the three learned policies

The short default run checks the complete pipeline; its numbers are not a meaningful architecture ranking. Use the same dataset split, seed, and step count for all three before comparing them. The tokenizer returns normalized actions, and open-loop evaluation converts them back to the dataset's raw units.

In [ ]:
names = ['factorized', 'autoregressive', 'parallel']
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for axis, metric, title in zip(
        axes, ['validation_ce', 'mae_std', 'jitter'],
        ['Held-out CE', 'MAE / training std', 'Sampled jitter']):
    axis.bar(names, [results[name][metric] for name in names])
    axis.set_title(title)
    axis.tick_params(axis='x', rotation=20)
fig.tight_layout()
plt.show()
for name in names:
    result = results[name]
    print(f"{name:14s} CE={result['validation_ce']:.3f}  "
          f"MAE/std={result['mae_std']:.3f}  "
          f"jitter={result['jitter']:.3f}")

## Next experiments

- Raise `TRAIN_STEPS` equally for all three heads before interpreting their diagnostic differences.
- Increase `samples` for smoother joint-support estimates; autoregressive sampling is intentionally slower.
- Compare chunk-by-chunk execution with `TemporalEnsembler` on validation episodes.
- Keep physical deployment separate: dataset units and the simulator/robot control mode must be converted and safety-checked explicitly.